# This notebook extracts CMG simulations results and save them as Numpy arrays
* File format conversion: CMG sr3/gmch.sr3 --> CMG rwo (by running CMG Results Report software on rwd files) --> numpy array. sr3 is the binary CMG simulation result file, whereas rwo is the extracted info in the ascii format. rwd is a configuration file to tell CMG Results Report software how to export results
* For Vienna goethermal, geomechanical simulations are stored in sr3 files, not in separate gmch.sr3 files.
* **This version uses `CMG2npy_robust`** instead of `CMG2npy`. The rwo file is read with a single `read()` rather than iterating ~700k lines, which avoids the silent partial reads seen when parsing off the Oak drive mapped over SMB (`Warning: Expected 139 values, got 76 ...`). Those rows used to be skipped and left as zeros; now any incomplete read raises instead.
* Set `expected_shape` and `expected_active` below so a short read is caught rather than passing silently.

# Step 0: Run this cell to provide inputs for all following steps

In [ ]:
from pathlib import Path

current_path = Path('.')

################## User Inputs ############################## 
name_prefix = 'test_260703' # file name prefix 
n_cases = 10 # number of simulation cases
property_list = ['STRESMXP','STRESMNP','STRESINT'] # list of properties (CMG keywords) to extract from the simulation results
sim_results_folder_path = current_path/f'{name_prefix}_dat_files' # path to the simulation results folder
sim_results_file_format = 'sr3' # sr3 or gmch.sr3
coor_fault_file_path = current_path/'JD_geothermal_coor&fault.npy' # path to the file containing grid cell coordinates and fault id
save_folder_path = sim_results_folder_path/f'{name_prefix}' # path to save extracted results
target_top_k = 5 # target top k layer to keep
target_bottom_k = 20 # target bottom k layer to keep

# integrity checks (used by CMG2npy_robust to catch an incomplete read)
expected_shape = (139, 248, 23) # (n_i, n_j, n_k) of the JD_geothermal grid
expected_active = 401735 # number of non-zero cells at the first time step, taken from a known-good case
copy_local_first = False # set True to copy each rwo to local disk before parsing, if the share is unreliable
wait_for_output = True # wait for each rwo to finish being written before moving on
################## End of User Inputs #######################

# Step 1: Convert CMG sr3/gmch.sr3 files to rwo files

Note this part needs to be run on a machine where CMG Results Report software is installed and the CMG2npy source code file.

In [ ]:
from pathlib import Path
from CMG2npy_robust import generate_CMG_rwd, run_CMG_rwd_report
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
# set in Step 0
################## End of User Inputs #######################

for property in property_list:
    for case_num in tqdm(range(1,n_cases+1),desc=f'Converting sr3 to rwo for keyword {property}'):
        generate_CMG_rwd(
            sr3_folder_path = sim_results_folder_path,
            case_name = f'case{case_num}',
            property = property,
            sim_results_file_format = sim_results_file_format,
            precision = 4
        )

        # passing `property` lets the function wait until the rwo file size stops
        # changing, so a write still sitting in the Windows client cache for the
        # mapped drive is not read too early
        run_CMG_rwd_report(
            rwd_folder_path = sim_results_folder_path,
            case_name = f'case{case_num}',
            property = property,
            cmg_version = 'ese-ts2win-v2024.20',
            wait_for_output = wait_for_output,
        )

        # remove rwd files
        Path(sim_results_folder_path, f'case{case_num}.rwd').unlink()

print("\nFinished generating rwo files for all cases.")

# Step 2: Extract simulation results from rwo files into numpy arrays

## Option 1: Extract results on all grid cells

In [ ]:
from pathlib import Path
from CMG2npy_robust import CMG_rwo2npy
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
# set in Step 0
################## End of User Inputs #######################

save_folder_path.mkdir(parents=True, exist_ok=True)

failed = []
for property in property_list:
    for case_num in tqdm(range(1,n_cases+1), desc=f'Generating numpy arrays for keyword {property}'):
        try:
            sim_results = CMG_rwo2npy(
                rwo_folder_path = sim_results_folder_path/'rwo',
                case_name = f'case{case_num}',
                property = property,
                is_save = True,
                save_folder_path = save_folder_path,
                show_info = False,
                expected_shape = expected_shape,
                expected_active = expected_active,
                copy_local_first = copy_local_first,
            )
        except Exception as e:
            # an incomplete read now raises instead of silently leaving zeros;
            # collect the failures so the whole batch does not stop on one case
            failed.append((case_num, property, str(e)))

print("\nFinished generating numpy arrays for all cases.")
if failed:
    print(f"\n{len(failed)} case/property combinations FAILED and were not saved:")
    for c,p,e in failed:
        print(f"  case{c} {p}: {e}")
    print("\nRe-run these; a repeated failure means the rwo itself is short (e.g. an early-terminated run).")

## Option 2: Only keep fault grid cells and simulated layers

- For the JD_geothermal grid (139, 248, 23), reservoir is between k=5 and 20.
- Note this part needs the file containing grid cell coordinates and fault id.

In [ ]:
import numpy as np
from pathlib import Path
from CMG2npy_robust import CMG_rwo2npy
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
# set in Step 0
################## End of User Inputs #######################

coor_fault = np.load(coor_fault_file_path) 
save_folder_path.mkdir(parents=True, exist_ok=True)

failed = []
for property in property_list:
    for case_num in tqdm(range(1,n_cases+1),desc=f'Generating numpy arrays for keyword {property}'):
        try:
            # extract results on all grid cells
            sim_results = CMG_rwo2npy(
                rwo_folder_path = sim_results_folder_path/'rwo',
                case_name = f'case{case_num}',
                property = property,
                is_save = False,
                save_folder_path = save_folder_path,
                show_info = False,
                expected_shape = expected_shape,
                expected_active = expected_active,
                copy_local_first = copy_local_first,
            )
        except Exception as e:
            failed.append((case_num, property, str(e)))
            continue

        # keep results on fault grid cells only
        nan_mask = np.isnan(coor_fault[:,:,:,3])
        sim_results[nan_mask] = np.nan

        # keep results of simulated layers only
        # Python indices are 0-based, for example, to extract slices 41-79, use 40:79
        sim_results_trimmed = sim_results[:, :, target_top_k-1:target_bottom_k]

        # save
        np.save(save_folder_path/f'case{case_num}_{property}.npy',sim_results_trimmed)


print("\nFinished generating numpy arrays for all cases.")
if failed:
    print(f"\n{len(failed)} case/property combinations FAILED and were not saved:")
    for c,p,e in failed:
        print(f"  case{c} {p}: {e}")

## Step 3: Check the extracted arrays before using them

Cases that terminated early (for example the thermal-front convergence failures) have fewer time steps than the rest. Downstream notebooks that assume a fixed `n_times` will fail on those, so list the shapes here first.

In [ ]:
import numpy as np
from pathlib import Path

################## User Inputs ############################## 
# set in Step 0
################## End of User Inputs #######################

shapes = {}
missing = []
for case_num in range(1,n_cases+1):
    f = save_folder_path/f'case{case_num}_{property_list[0]}.npy'
    if not f.exists():
        missing.append(case_num); continue
    a = np.load(f, mmap_mode='r')
    shapes.setdefault(a.shape, []).append(case_num)

print(f'Shapes found in {save_folder_path}:')
for s, cases in sorted(shapes.items(), key=lambda kv: -len(kv[1])):
    print(f'  {s}: {len(cases)} cases  e.g. {cases[:8]}')
if missing:
    print(f'\nMISSING (not extracted): {missing}')

if len(shapes) > 1:
    n_times_ref = max(shapes, key=lambda s: len(shapes[s]))[-1]
    odd = [c for s, cases in shapes.items() if s[-1] != n_times_ref for c in cases]
    print(f'\nWARNING: cases with a different number of time steps: {sorted(odd)}')
    print('Exclude these downstream, or derive n_times per case instead of fixing it.')
else:
    print('\nAll extracted cases share the same shape.')